# Data Visualization

Great visualizations tell stories that tables cannot. This notebook builds a practical toolkit for matplotlib, seaborn, and plotly.

**Rule:** Choose the chart type based on what relationship you're showing, not because it looks cool.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

# Load datasets
iris = sns.load_dataset('iris')
tips = sns.load_dataset('tips')
titanic = sns.load_dataset('titanic')
print('Datasets loaded: iris, tips, titanic')

## 1. Chart Type Guide

| Relationship | Chart Type |
|-------------|------------|
| Distribution of one variable | Histogram, KDE, Box plot, Violin |
| Two numeric variables | Scatter plot, Hex bin |
| Category vs numeric | Bar chart, Box plot, Violin |
| Composition | Pie/donut, Stacked bar, Treemap |
| Time series | Line chart |
| Many variables | Pair plot, Heatmap (correlation) |
| Geographic | Choropleth, Dot map |

In [ ]:
# Distribution charts gallery
data = np.random.exponential(scale=2, size=500)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Histogram
axes[0].hist(data, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Histogram'); axes[0].set_xlabel('Value')

# KDE (smoothed histogram)
axes[1].hist(data, bins=30, density=True, color='steelblue', alpha=0.4)
x_kde = np.linspace(data.min(), data.max(), 200)
kde = stats.gaussian_kde(data)
axes[1].plot(x_kde, kde(x_kde), 'r-', lw=2)
axes[1].set_title('Histogram + KDE'); axes[1].set_xlabel('Value')

# Box plot
groups = {'A': np.random.normal(5, 1.5, 100), 'B': np.random.normal(6, 2, 100), 'C': np.random.normal(4, 1, 100)}
axes[2].boxplot(list(groups.values()), labels=list(groups.keys()), patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[2].set_title('Box Plot'); axes[2].set_ylabel('Value')

# Violin plot
axes[3].violinplot(list(groups.values()), positions=[1,2,3], showmedians=True)
axes[3].set_xticks([1,2,3]); axes[3].set_xticklabels(['A','B','C'])
axes[3].set_title('Violin Plot'); axes[3].set_ylabel('Value')

plt.suptitle('Distribution Charts', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Seaborn statistical plots — the DS workhorse
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Scatter with hue and size
sns.scatterplot(data=iris, x='sepal_length', y='petal_length', 
                hue='species', size='petal_width', alpha=0.7, ax=axes[0,0])
axes[0,0].set_title('Scatter: sepal vs petal length')

# Violin with inner boxplot
sns.violinplot(data=tips, x='day', y='total_bill', hue='sex', 
               split=True, ax=axes[0,1], palette='husl')
axes[0,1].set_title('Violin: Tips by Day & Sex')

# Heatmap: correlation matrix
corr = iris.drop('species', axis=1).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=axes[0,2], square=True)
axes[0,2].set_title('Correlation Heatmap')

# Regression plot with CI
sns.regplot(data=tips, x='total_bill', y='tip', ax=axes[1,0], 
            scatter_kws={'alpha':0.4}, line_kws={'color':'red'})
axes[1,0].set_title('Regression: Bill vs Tip')

# Count plot
sns.countplot(data=titanic, x='class', hue='survived', ax=axes[1,1],
              palette={'0':'steelblue', '1':'darkorange'})
axes[1,1].set_title('Bar: Titanic Survival by Class')
axes[1,1].legend(title='Survived', labels=['No','Yes'])

# KDE plot
for species in iris['species'].unique():
    subset = iris[iris['species'] == species]['sepal_length']
    axes[1,2].hist(subset, bins=15, alpha=0.5, density=True, label=species)
axes[1,2].set_title('Overlapping KDE: Sepal Length by Species')
axes[1,2].legend()

plt.suptitle('Seaborn Statistical Visualization Gallery', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Pair plot — essential for multivariate EDA
g = sns.pairplot(iris, hue='species', diag_kind='kde',
                 plot_kws={'alpha': 0.6, 's': 20})
g.fig.suptitle('Pair Plot: All Pairwise Relationships in Iris', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# matplotlib anatomy: publication-quality figure
np.random.seed(42)
x = np.linspace(0, 10, 100)
methods = {
    'Model A': x**0.5 + np.random.randn(100)*0.2,
    'Model B': 0.8*x**0.5 + np.random.randn(100)*0.3,
    'Baseline': np.linspace(0, 2.5, 100),
}
colors = ['steelblue', 'darkorange', 'gray']

fig, ax = plt.subplots(figsize=(10, 6))

for (name, values), color in zip(methods.items(), colors):
    ax.plot(x, values, color=color, lw=2, label=name)
    if name != 'Baseline':
        ci = 0.3
        ax.fill_between(x, values-ci, values+ci, color=color, alpha=0.15)

# Annotations
ax.annotate('Model A outperforms\nbaseline here', xy=(8, 2.0), xytext=(5, 2.8),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=11, ha='center')

ax.set_xlabel('Training Steps', fontsize=13)
ax.set_ylabel('Performance Score', fontsize=13)
ax.set_title('Model Performance Comparison\n(with 95% Confidence Intervals)', fontsize=14)
ax.legend(title='Methods', fontsize=11, title_fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()
print('\nTip: Save high-quality figures with: plt.savefig("fig.pdf", dpi=300, bbox_inches="tight")')